# 03 - Baseline MLP (Keras) e Conversão hls4ml

Este notebook substitui o modelo Ridge por um Multi-Layer Perceptron (MLP) utilizando TensorFlow/Keras. O objetivo é treinar uma rede neural simples compatível com o fluxo do hls4ml para posterior síntese em FPGA.

Etapas:
1. Treinamento da MLP em Keras.
2. Geração do projeto C++ via hls4ml.
3. Compilação e simulação em software (C-Simulation).
4. Síntese de hardware (requer Vivado HLS).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.optimizers import Adam
import hls4ml

sys.path.append(str(Path("..").resolve() / "src"))
from emg_hls4ml_mvp.dataset import build_feature_dataset

## 1. Aquisição e Preparação dos Dados

In [ ]:
X, y, _ = build_feature_dataset(
    data_root=Path("../data"),
    subject="Sub001",
    recording_id="Sub001_1_05_450_0",
    target_column="index_z",
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 2. Treinamento do Modelo MLP

In [ ]:
model = Sequential()
model.add(Dense(32, input_shape=(X_train_scaled.shape[1],), name="fc1"))
model.add(Activation("relu", name="relu1"))
model.add(Dense(1, name="output"))
model.add(Activation("linear", name="linear_out"))

model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")

print("Treinando modelo Keras...")
history = model.fit(X_train_scaled, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=1)

keras_pred = model.predict(X_test_scaled).flatten()
print(f"\nR2 Score (Keras): {r2_score(y_test, keras_pred):.4f}")

## 3. Conversão para hls4ml e Simulação C++

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity="model")
print("Configuração hls4ml:", config)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir="model_1/hls4ml_prj",
    part="xczu7ev-ffvc1156-2-e", # Target FPGA
    clock_period=10.0
)

# Compila a simulação em C++.
# Nota: Em ambientes macOS com Apple Clang recente, isso pode falhar devido a conflitos
# de namespace na biblioteca complex.h. Recomenda-se rodar em ambiente Linux.
hls_model.compile()

hls_pred = hls_model.predict(X_test_scaled).flatten()
print(f"MSE entre Keras (float) e HLS (fixed-point): {mean_squared_error(keras_pred, hls_pred):.6f}")

## 4. Síntese de Hardware (Requer Vivado HLS)
Esta etapa executa o Vivado HLS para converter o código C++ em RTL (Verilog/VHDL) e gera os relatórios de síntese detalhando o uso de recursos (LUTs, DSPs) e a estimativa de latência.

In [ ]:
# Descomente a linha abaixo e execute no servidor Linux com o Vivado instalado.
# hls_model.build(csim=False, synth=True, vsynth=True)